---
## PumpKin: A machine learning package for automatically tracking pharyngeal pumping kinematics in freely moving *C. elegans*
Run PumpKin on all videos to obtain detected pump times and estimates of the continuous pumping rate. **Make sure that you have run `EZ_FRCNN.ipynb` to track the grinder and `saveCoMs.ipynb` to obtain grinder CoMs first!**

In [ ]:
### Import libraries + define paths
from library import *

VID_DIR = './videos/' # Directory containing videos to be analyzed

### Get names of all videos within the folder
video_type  = '.wmv' # CHANGE FOR YOUR DATA
video_paths = glob.glob(VID_DIR + '*' + video_type)  
videos      = [os.path.splitext(os.path.basename(v))[0] for v in video_paths]

In [ ]:
### Process all videos
all_motions, all_rates, all_peaks, all_troughs = [], [], [], []

### Run PumpKin on all videos to be analyzed
for video in videos:
    ### Load tracked grinder CoMs
    com_name  = VID_DIR + '/outputs/coms/' + video + '_coms.csv'
    coms      = np.loadtxt(com_name)
    
    ### Set paths for saving (OPTIONAL) and for video
    full_name  = video + '_cropped' + video_type
    video_path = VID_DIR + '/cropped/' + full_name
    save_path  = VID_DIR + '/outputs/pumpkin/'
    vid        = cv2.VideoCapture(video_path)
    fps        = vid.get(cv2.CAP_PROP_FPS)

    ### Get raw grinder motion from video
    print('Processing ' + video)
    mags, angs = process_video(video_path, coms, save=True, save_path=save_path)
    
    ### Get filtered grinder motion
    print('Filtering grinder motion for ' + video)
    strain_name, cond_name = get_strain_cond(full_name)
    grinder_motion, fc = get_filtered_motion(strain_name, cond_name, mags, angs, fps)
    all_motions.append(grinder_motion)
    
    ### Get pumping rate (and pump times)
    print('Obtaining pumping rate for ' + video)
    pr_cont, pr_disc, peaks, troughs = get_pumping_rate(video_path, grinder_motion, fps, fc, save=True, save_path=save_path)
    all_rates.append(pr_cont)
    all_peaks.append(peaks)
    all_troughs.append(troughs)
    
print('Finished processing all videos.')

In [ ]:
### Generate plots for all videos
plt.rcParams['axes.grid'] = False
for i, video in enumerate(videos):
    grinder_motion = all_motions[i]
    troughs        = all_troughs[i]
    peaks          = all_peaks[i]
    pk_pr_cont     = all_rates[i]

    ######################################################################
    ### Grinder motion with peaks and troughs labeled
    dt     = 1/fps
    time_g = np.arange(0,(len(grinder_motion))*dt,dt)
    save_path = './plots/' + videos[i] + '_PumpKin_motion.svg'
    plt.figure(figsize=(6,2))
    plt.plot(time_g,             grinder_motion, color='lightcoral', zorder=0)
    plt.scatter(time_g[troughs], grinder_motion[troughs], color='maroon',   s=20, zorder=1)  # Red dots for troughs
    plt.scatter(time_g[peaks],   grinder_motion[peaks],   color='seagreen', s=20, zorder=2)  # Green dots for peaks
    plt.xlabel('Time (s)')
    plt.ylabel('Motion amplitude (px)')
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()
    
    ######################################################################
    ### Pump raster plot
    save_path = './plots/' + videos[i] + '_PumpKin_raster.svg'
    plt.figure(figsize=(6,1))
    plt.eventplot(all_troughs[i]/fps, lw=3, linelengths=0.5, color='maroon')
    plt.tick_params(labelleft=False, left=False)
    plt.xlabel('Time (s)')
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()

    ######################################################################
    ### Continuous pumping rate
    save_path = './plots/' + videos[i] + '_PumpKin_rate.svg'
    plt.figure(figsize=(6,2))
    plt.plot(pk_pr_cont,color='orange',lw=2,label='PumpKin (mean = ' + str(np.round(np.mean(pk_pr_cont),2)) + ')')
    plt.xlabel("Time (s)")
    plt.ylabel("Pumping rate (pumps / second)")
    plt.savefig(save_path, bbox_inches="tight")
    plt.show()